In [ ]:
# =============================================================================
#  FPT STOCK WEEKLY TREND CLASSIFICATION — CAUSAL MEDIAN FILTER FEATURES
#  2-CLASS VERSION: Up / Down
#
#  KIẾN TRÚC TỔNG QUAN:
#    Input  : Weekly OHLCV features from FPT stock data
#    Phase 1: RAW technical-feature baseline
#             → tạo weekly OHLCV
#             → tạo các feature kỹ thuật từ close/high/low/open/volume
#             → chronological train/test split
#
#    Phase 2: MEDIAN-FILTERED stock features
#             → causal rolling median trên close price
#             → tạo thêm fitted/residual/noise features
#             → train cùng classifier và cùng chronological split như baseline
#
#  TẠI SAO DÙNG CAUSAL MEDIAN FILTER CHO STOCK?
#    - Với stock prediction, không được dùng centered median vì sẽ leak future data
#    - Causal median chỉ dùng giá hiện tại và quá khứ tại thời điểm t
#    - Mục tiêu không phải “làm sạch tín hiệu ECG”, mà là tạo thêm feature mô tả
#      local trend và residual noise của chuỗi giá
# =============================================================================

# # FPT Stock Weekly Trend Classification — Causal Median Filter Experiment
# 
# This notebook adds a **causal rolling median filter** for FPT close-price denoising.
# 
# Experiment design:
# 
# - **P1 Baseline**: weekly technical features from raw FPT price/volume
# - **P2 Median**: same baseline features + causal median-filter features
# 
# For stock prediction, the median filter must be **causal**: value at time `t` uses only current and previous prices. A centered median filter would leak future information.

# =============================================================================
# SETUP
# =============================================================================
from __future__ import annotations

import os, sys
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, VotingClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.pipeline import make_pipeline

def find_project_root() -> Path:
    candidates = [Path.cwd(), *Path.cwd().parents]
    for c in candidates:
        if (c / "src" / "model" / "stock").exists():
            return c
    return Path("/kaggle/working/IT2022_Projects-main")

PROJECT_ROOT = find_project_root()
sys.path.insert(0, str(PROJECT_ROOT))

from src.model.stock.median_denoiser import StockMedianDenoiser, StockMedianDenoiserConfig

OUT_DIR = PROJECT_ROOT / "outputs" / "stock_median"
OUT_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
np.random.seed(SEED)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("OUT_DIR:", OUT_DIR)

# =============================================================================
# DATA PATH
# =============================================================================
def resolve_fpt_path() -> Path:
    candidates = [
        PROJECT_ROOT / "src" / "data" / "stock" / "FPT raw.csv",
        Path("/kaggle/input/datasets/cdnghnam/fptckhon/FPT raw.csv"),
        Path("/kaggle/input/fptckhon/FPT raw.csv"),
    ]
    for c in candidates:
        if c.exists():
            return c
    raise FileNotFoundError("Could not find FPT raw.csv. Update resolve_fpt_path().")

DATA_PATH_FPT = resolve_fpt_path()
print("DATA_PATH_FPT:", DATA_PATH_FPT)

# =============================================================================
# LOAD DAILY DATA AND CREATE WEEKLY OHLCV
# =============================================================================
def load_daily(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path)
    df["Date"] = pd.to_datetime(df["Date"])
    df = df.sort_values("Date").reset_index(drop=True)
    for c in df.columns:
        if c != "Date":
            df[c] = pd.to_numeric(df[c], errors="coerce")
    return df

def make_weekly(df: pd.DataFrame) -> pd.DataFrame:
    d = df.set_index("Date")
    need = ["Open_FPT", "High_FPT", "Low_FPT", "Close_FPT", "Volume_FPT"]
    missing = [c for c in need if c not in d.columns]
    if missing:
        raise KeyError(f"Missing expected FPT columns: {missing}")
    w = pd.DataFrame({
        "open": d["Open_FPT"].resample("W-FRI").first(),
        "high": d["High_FPT"].resample("W-FRI").max(),
        "low": d["Low_FPT"].resample("W-FRI").min(),
        "close": d["Close_FPT"].resample("W-FRI").last(),
        "volume": d["Volume_FPT"].resample("W-FRI").sum(),
    }).dropna().reset_index()
    return w

df_daily = load_daily(DATA_PATH_FPT)
df_w = make_weekly(df_daily)

print(df_daily.shape, df_w.shape)
df_w.head()

# =============================================================================
# LABEL NEXT-WEEK DIRECTION
# =============================================================================
# At week t, features use information up to week t.
# The label is whether close(t+1) is higher or lower than close(t).

SIGNAL_THR = 0.0  # set to 1.0 or 2.0 if you want to drop sideways weeks

def label_weekly(df_w: pd.DataFrame, signal_thr: float = 0.0) -> pd.DataFrame:
    df = df_w.copy()
    df["future_ret"] = df["close"].shift(-1) / df["close"] - 1.0
    df["future_ret_pct"] = df["future_ret"] * 100
    df["label"] = np.where(df["future_ret_pct"] > signal_thr, "up",
                    np.where(df["future_ret_pct"] < -signal_thr, "down", "sideway"))
    df = df[df["label"] != "sideway"].dropna().reset_index(drop=True)
    return df

df_lab = label_weekly(df_w, SIGNAL_THR)
print(df_lab["label"].value_counts())
df_lab[["Date", "close", "future_ret_pct", "label"]].head()

# =============================================================================
# BASELINE WEEKLY FEATURES
# =============================================================================
def build_base_features(df: pd.DataFrame) -> pd.DataFrame:
    f = pd.DataFrame(index=df.index)
    close = df["close"].astype(float)
    high = df["high"].astype(float)
    low = df["low"].astype(float)
    open_ = df["open"].astype(float)
    volume = df["volume"].astype(float)

    f["ret_1"] = close.pct_change(1) * 100
    f["ret_2"] = close.pct_change(2) * 100
    f["ret_4"] = close.pct_change(4) * 100
    f["range_pct"] = (high - low) / close.replace(0, np.nan) * 100
    f["body_pct"] = (close - open_) / open_.replace(0, np.nan) * 100
    f["vol_chg_1"] = volume.pct_change(1) * 100

    for w in [3, 5, 10]:
        ma = close.rolling(w, min_periods=1).mean()
        sd = close.rolling(w, min_periods=2).std()
        f[f"ma_gap_{w}"] = (close - ma) / close.replace(0, np.nan) * 100
        f[f"roll_std_{w}"] = sd.fillna(0.0)
        f[f"vol_ma_gap_{w}"] = (volume - volume.rolling(w, min_periods=1).mean()) / volume.replace(0, np.nan) * 100

    return f.replace([np.inf, -np.inf], np.nan).ffill().bfill().fillna(0.0)

X_base_df = build_base_features(df_lab)
X_base_df.head()

# =============================================================================
# CAUSAL MEDIAN FILTER FEATURES
# =============================================================================
MEDIAN_WINDOW = 5

cfg = StockMedianDenoiserConfig(window=MEDIAN_WINDOW, causal=True)
denoiser = StockMedianDenoiser(cfg)
denoiser.fit(df_lab["close"].values)
X_med_df, med_price, med_resid, _ = denoiser.transform(df_lab["close"].values)

X_med_df = X_med_df.replace([np.inf, -np.inf], np.nan).ffill().bfill().fillna(0.0)
X_all_med_df = pd.concat([X_base_df.reset_index(drop=True), X_med_df.reset_index(drop=True)], axis=1)

print("Base feature count:", X_base_df.shape[1])
print("Median feature count:", X_med_df.shape[1])
print("Combined feature count:", X_all_med_df.shape[1])
X_med_df.head()

# =============================================================================
# VISUAL CHECK
# =============================================================================
plt.figure(figsize=(11, 4))
plt.plot(df_lab["Date"], df_lab["close"], label="Raw weekly close", linewidth=1.5)
plt.plot(df_lab["Date"], med_price, label=f"Causal median close, window={MEDIAN_WINDOW}", linewidth=1.5)
plt.title("FPT weekly close before vs after causal median filtering")
plt.xlabel("Date")
plt.ylabel("Close")
plt.legend()
plt.tight_layout()
plt.savefig(OUT_DIR / "stock_median_close.png", dpi=160)
plt.show()

# =============================================================================
# WALK-FORWARD-LIKE TIME SPLIT
# =============================================================================
# Simple chronological split: train on early weeks, test on later weeks.
# This avoids random shuffling, which would leak time information in financial data.

le = LabelEncoder()
y = le.fit_transform(df_lab["label"].values)
label_list = list(le.classes_)

N = len(df_lab)
split = int(N * 0.75)

X_base = X_base_df.values.astype(float)
X_med = X_all_med_df.values.astype(float)

Xb_tr, Xb_te = X_base[:split], X_base[split:]
Xm_tr, Xm_te = X_med[:split], X_med[split:]
y_tr, y_te = y[:split], y[split:]

print("Labels:", label_list)
print("Train/test:", len(y_tr), len(y_te))
print("Test period:", df_lab.loc[split, "Date"], "to", df_lab.loc[len(df_lab)-1, "Date"])

# =============================================================================
# TRAIN SAME CLASSIFIER FOR P1 AND P2
# =============================================================================
def build_classifier():
    rf = RandomForestClassifier(
        n_estimators=400,
        max_depth=6,
        min_samples_leaf=4,
        class_weight="balanced",
        random_state=SEED,
        n_jobs=-1,
    )
    et = ExtraTreesClassifier(
        n_estimators=400,
        max_depth=6,
        min_samples_leaf=4,
        class_weight="balanced",
        random_state=SEED + 1,
        n_jobs=-1,
    )
    return make_pipeline(
        StandardScaler(),
        VotingClassifier([("rf", rf), ("et", et)], voting="soft")
    )

def train_eval(name, X_train, X_test):
    clf = build_classifier()
    clf.fit(X_train, y_tr)
    pred = clf.predict(X_test)
    acc = accuracy_score(y_te, pred)
    f1 = f1_score(y_te, pred, average="macro", zero_division=0)
    print(f"[{name}] acc={acc:.4f} macro_f1={f1:.4f}")
    print(classification_report(y_te, pred, target_names=label_list, zero_division=0))
    return clf, pred, {"accuracy": acc, "macro_f1": f1}

base_clf, base_pred, base_metrics = train_eval("P1_BASELINE", Xb_tr, Xb_te)
med_clf, med_pred, med_metrics = train_eval(f"P2_MEDIAN_w{MEDIAN_WINDOW}", Xm_tr, Xm_te)

# =============================================================================
# RESULT TABLE
# =============================================================================
results = pd.DataFrame([
    {"phase": "P1_BASELINE", **base_metrics},
    {"phase": f"P2_MEDIAN_w{MEDIAN_WINDOW}", **med_metrics},
])
results.to_csv(OUT_DIR / "stock_median_results.csv", index=False)
results

# =============================================================================
# CONFUSION MATRICES
# =============================================================================
def plot_cm(y_true, y_pred, title, filename):
    cm = confusion_matrix(y_true, y_pred, labels=np.arange(len(label_list)))
    fig, ax = plt.subplots(figsize=(4.5, 4))
    im = ax.imshow(cm)
    ax.set_title(title)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_xticks(range(len(label_list)))
    ax.set_yticks(range(len(label_list)))
    ax.set_xticklabels(label_list)
    ax.set_yticklabels(label_list)
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, str(cm[i, j]), ha="center", va="center")
    fig.colorbar(im, ax=ax)
    plt.tight_layout()
    plt.savefig(OUT_DIR / filename, dpi=160)
    plt.show()

plot_cm(y_te, base_pred, "P1 Baseline confusion matrix", "cm_stock_baseline.png")
plot_cm(y_te, med_pred, f"P2 Median window={MEDIAN_WINDOW} confusion matrix", "cm_stock_median.png")

# ## Report wording
# 
# For the stock task, median filtering is used as a **causal rolling median** over weekly close prices. It produces denoised-price features such as fitted value, residual, residual magnitude, trend, curvature, fitted return, and noise ratio. These are added to the baseline technical features and evaluated with the same chronological train/test split. The causal setting is necessary because stock prediction must not use future prices.
